In [0]:
import dlt
from pyspark.sql.functions import current_timestamp

# Configurable parameter set at pipeline level (e.g., dev, stage, prod)
input_path = dlt.config("input.path")

@dlt.table(name="bronze.default.ito_sensor")
def load_raw_data():
    return (
        spark.read.option("header", "true")
        .csv(input_path)
        .withColumn("ingestion_time", current_timestamp())
    )

In [0]:
import pyspark.sql.functions as F
def clean_and_standardize(df):
    return (
        df.withColumn("age", F.col("age").cast("int"))
          .withColumn("cholesterol", F.col("cholesterol").cast("double"))
          .filter(F.col("age").isNotNull() & F.col("cholesterol").isNotNull())
    )

@dlt.table(name="bronze.default.ito_sensor")
def transform_to_silver():
    df = dlt.read("bronze.default.ito_sensor")
    return clean_and_standardize(df)